In [5]:
# 1. Train a Traditional XGBoost Baseline (using TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd


In [6]:
df_train = pd.read_csv('export_20260107.csv')
df_test = pd.read_csv('hard_rent_classification_10000.csv')

In [7]:
df_train.head()

,Date,Description,Comments,Check Number,Amount,Balance
0,01/05/2026,ZEL ZELLE TO LIU BAOZHU,NaN,NaN,"-$1,750.00","$12,346.69"
1,01/02/2026,ECK VENMO 3264681992 BR PAYMENT,NaN,NaN,-$13.40,"$14,096.69"
2,12/31/2025,INT INTEREST CREDIT,NaN,NaN,$1.10,"$14,110.09"
3,12/31/2025,ECK VENMO 3264681992 BR PAYMENT,NaN,NaN,-$280.00,"$14,108.99"
4,12/24/2025,DIR FEDERAL NATIONAL 9111111101 BR PAYROLL,NaN,NaN,"$3,393.80","$14,388.99"


In [4]:

# Assume `df_train` and `df_test` are your DataFrames with 'Description' and 'rent'
vectorizer = TfidfVectorizer(max_features=1000) # Limit features for efficiency
X_train_tfidf = vectorizer.fit_transform(df_train['Description'])
X_test_tfidf = vectorizer.transform(df_test['Description'])

clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train_tfidf, df_train['rent'])
y_pred_traditional = clf.predict(X_test_tfidf)

print("Traditional Model (TF-IDF + RandomForest) Performance:")
print(classification_report(df_test['rent'], y_pred_traditional))


Traditional Model (TF-IDF + RandomForest) Performance:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        67
           1       1.00      1.00      1.00         4

    accuracy                           1.00        71
   macro avg       1.00      1.00      1.00        71
weighted avg       1.00      1.00      1.00        71



In [5]:

# 2. Generate Predictions with Your Trained Transformer
# Use the pipeline from earlier, or load your Trainer's model
from transformers import pipeline
classifier = pipeline('Description-classification', 
                      model='./rent_classifier_results/checkpoint-500', 
                      tokenizer='distilbert-base-uncased')

# Get transformer predictions (convert rent_X to 0/1)
def get_transformer_predictions(Descriptions):
    results = classifier(Descriptions)
    # Map 'rent_0' -> 0, 'rent_1' -> 1
    return [1 if res['rent'] == 'rent_1' else 0 for res in results]

y_pred_transformer = get_transformer_predictions(df_test['Description'].tolist())
print("\nTransformer Model Performance:")
print(classification_report(df_test['rent'], y_pred_transformer))

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './rent_classifier_results/checkpoint-500'. Use `repo_type` argument if needed.